In [5]:
import pandas as pd
import numpy as np
from statistics import mean
from scipy.stats import invgamma

- The first thing we do is read the files we have from Github, and process them
- These files are in a folder called "Downloaded Files"
- For each season, we created a big file, which has for each player, all the gws data from that season
- Once we have processed them, we save the new files in a folder called "Modified season files"
- So then after this python file, we buld the mcmc model, and then we use this past github data to optimsie the model parameters (MCMC model). Then once we have the model we use fpl api (Player prediction data) to then run predictions on this seasons data

In [6]:
# Function to process each season
def process_data(merged_gw, teams, team_quality, opponent_quality, output_name):
    # add .csv 
    merged_gw_filename = merged_gw + ".csv"
    teams_filename = teams + ".csv"
    output_filename = output_name + ".csv"

    
    # Load initial downlaoded gw data from github
    player_info_file_path = f'/Users/alexroberts/Documents/Diss (new)/(1) Making csv/Downloaded Files/{merged_gw_filename}'
    data = pd.read_csv(player_info_file_path)

    # Load initial downlaoded teams data from github
    teams_file_path = f'/Users/alexroberts/Documents/Diss (new)/(1) Making csv/Downloaded Files/{teams_filename}'
    teams_df = pd.read_csv(teams_file_path)


    # columns to keep
    columns_to_keep = [
        'name', 'GW', 'position', 'team', 'opponent_team',
        'was_home', 'minutes', 'value', 'influence', 
        'creativity', 'threat', 'total_points'
    ]
    data = data[columns_to_keep]
    
    # Change 'was_home' from TRUE/FALSE to integer
    data['was_home'] = data['was_home'].astype(int)

    
    # Merge the datasets to replace opponent_team ID with the team name
    data = data.merge(teams_df[['id', 'name']], how='left', 
                      left_on='opponent_team', right_on='id')
    

    # Merge to get actual team name
    if teams == "teams20":
        data = data.merge(teams_df[['id', 'name']], how='left',
                        left_on='team', right_on='id')
        data['team'] = data['name']
        data = data.drop(columns=['name', 'id_x', 'id_y'])
    else:
        data = data.drop(columns=['id'])
        
    
    # Replace the 'opponent_team' column with the team 'name' from the teams dataset
    data['opponent_team'] = data['name_y']
    
    # Drop the extra columns from the merge
    data = data.drop(columns=['name_y'])
    
    # Map 'team' and 'opponent_team' to quality
    data['team'] = data['team'].map(team_quality)
    data['opponent_team'] = data['opponent_team'].map(opponent_quality)
    
    # Rename 'name' columns
    if 'name_x' in data.columns:
        data.rename(columns={'name_x': 'name'}, inplace=True)

        
    # set output path names
    output_path = f'/Users/alexroberts/Documents/Diss (new)/(1) Making csv/Modified season files/{output_filename}'
    data.to_csv(output_path, index=False)
    
    print('Done')

In [7]:
# Quality Rankings
#######################

#######################
# 24
quality_24 = {
    "Man City": 1,
    "Arsenal": 2,
    "Man Utd": 3,
    "Newcastle": 4,
    "Liverpool": 5,
    "Brighton": 6,
    "Aston Villa": 7,
    "Spurs": 8,
    "Brentford": 9,
    "Fulham": 10,
    "Crystal Palace": 11,
    "Chelsea": 12,
    "Wolves": 13,
    "West Ham": 14,
    "Bournemouth": 15,
    "Nott'm Forest": 16,
    "Everton": 17,
    "Burnley": 18,
    "Sheffield Utd": 19,
    "Luton": 20
}


#######################
# 23
quality_23 = {
    "Man City": 1,
    "Liverpool": 2,
    "Chelsea": 3,
    "Spurs": 4,
    "Arsenal": 5,
    "Man Utd": 6,
    "West Ham": 7,
    "Leicester": 8,
    "Brighton": 9,
    "Wolves": 10,
    "Newcastle": 11,
    "Crystal Palace": 12,
    "Brentford": 13,
    "Aston Villa": 14,
    "Southampton": 15,
    "Everton": 16,
    "Leeds": 17,
    "Fulham": 18,
    "Bournemouth": 19,
    "Nottm Forest": 20
}


#######################
# 22
opponent_quality_22 = {
    "Manchester City": 1,
    "Manchester Utd": 2,
    "Liverpool": 3,
    "Chelsea": 4,
    "Leicester City": 5,
    "West Ham": 6,
    "Tottenham": 7,
    "Arsenal": 8,
    "Leeds United": 9,
    "Everton": 10,
    "Aston Villa": 11,
    "Newcastle": 12,
    "Wolves": 13,
    "Crystal Palace": 14,
    "Southampton": 15,
    "Brighton": 16,
    "Burnley": 17,
    "Norwich City": 18,
    "Watford": 19,
    "Brentford": 20
}


team_quality_22 = {
    "Man City": 1,
    "Man Utd": 2,
    "Liverpool": 3,
    "Chelsea": 4,
    "Leicester": 5,
    "West Ham": 6,
    "Spurs": 7,
    "Arsenal": 8,
    "Leeds": 9,
    "Everton": 10,
    "Aston Villa": 11,
    "Newcastle": 12,
    "Wolves": 13,
    "Crystal Palace": 14,
    "Southampton": 15,
    "Brighton": 16,
    "Burnley": 17,
    "Norwich": 18,
    "Watford": 19,
    "Brentford": 20
}


#######################
# 21
opponent_quality_21 = {
    "Liverpool": 1,
    "Man City": 2,
    "Man Utd": 3,
    "Chelsea": 4,
    "Leicester": 5,
    "Spurs": 6,
    "Wolves": 7,
    "Arsenal": 8,
    "Sheffield Utd": 9,
    "Burnley": 10,
    "Southampton": 11,
    "Everton": 12,
    "Newcastle": 13,
    "Crystal Palace": 14,
    "Brighton": 15,
    "West Ham": 16,
    "Aston Villa": 17,
    "Leeds": 18,
    "West Brom": 19,
    "Fulham": 20
}


team_quality_21 = {
    "Liverpool": 1,
    "Man City": 2,
    "Man Utd": 3,
    "Chelsea": 4,
    "Leicester": 5,
    "Spurs": 6,
    "Wolves": 7,
    "Arsenal": 8,
    "Sheffield Utd": 9,
    "Burnley": 10,
    "Southampton": 11,
    "Everton": 12,
    "Newcastle": 13,
    "Crystal Palace": 14,
    "Brighton": 15,
    "West Ham": 16,
    "Aston Villa": 17,
    "Leeds": 18,
    "West Brom": 19,
    "Fulham": 20
}


In [8]:
process_data('merged_gw_24', 'teams24', quality_24, quality_24, 'data_24')
process_data('merged_gw_23', 'teams23', quality_23, quality_23, 'data_23')
process_data('merged_gw_22', 'teams22', team_quality_22, opponent_quality_22, 'data_22')
process_data('merged_gw_21', 'teams21', team_quality_21, opponent_quality_21, 'data_21')

Done
Done
Done
Done


In [9]:
data_21 = pd.read_csv('/Users/alexroberts/Documents/Diss (new)/(1) Making csv/Modified season files/data_21.csv')
data_22 = pd.read_csv('/Users/alexroberts/Documents/Diss (new)/(1) Making csv/Modified season files/data_22.csv')
data_23 = pd.read_csv('/Users/alexroberts/Documents/Diss (new)/(1) Making csv/Modified season files/data_23.csv')
data_24 = pd.read_csv('/Users/alexroberts/Documents/Diss (new)/(1) Making csv/Modified season files/data_24.csv')

# change GW 
data_22['GW'] += 38 
data_23['GW'] += 76 
data_24['GW'] += 114

# Combine all three filtered datasets
combined_data = pd.concat([data_21, data_22, data_23, data_24], ignore_index=True)
data = combined_data



positions = ["GK", "DEF", "MID", "FWD"]

# EWMA and rolling config
decay_span = 5
rolling_window = 20

# Define columns to save in per-GW files
subset_cols = ["name", "team", "opponent_team", "value", "was_home", "total_points", "minutes",
               "ewma_total_points", "ewma_influence", "ewma_creativity", "ewma_threat",
               "avg_total_points_20", "avg_influence_20", "avg_creativity_20", "avg_threat_20"]

# Iterate through each position
for position in positions:
    # Filter data for the current position
    position_data = data[data['position'] == position]

    # Store data
    all_players_data = []

    # Iterate through each player
    for player in position_data['name'].unique():
        player_data = position_data[position_data['name'] == player].copy()
        
        # Skip players with low points
        if player_data['total_points'].sum() < 10:
            continue  

        # Calculate EWMA and rolling averages
        player_data['ewma_total_points'] = player_data['total_points'].shift(1).ewm(span=decay_span, adjust=False).mean().round(2)
        player_data['ewma_influence'] = player_data['influence'].shift(1).ewm(span=decay_span, adjust=False).mean().round(2)
        player_data['ewma_creativity'] = player_data['creativity'].shift(1).ewm(span=decay_span, adjust=False).mean().round(2)
        player_data['ewma_threat'] = player_data['threat'].shift(1).ewm(span=decay_span, adjust=False).mean().round(2)

        player_data['avg_total_points_20'] = player_data['total_points'].shift(1).rolling(window=rolling_window, min_periods=1).mean().round(2)
        player_data['avg_influence_20'] = player_data['influence'].shift(1).rolling(window=rolling_window, min_periods=1).mean().round(2)
        player_data['avg_creativity_20'] = player_data['creativity'].shift(1).rolling(window=rolling_window, min_periods=1).mean().round(2)
        player_data['avg_threat_20'] = player_data['threat'].shift(1).rolling(window=rolling_window, min_periods=1).mean().round(2)

        all_players_data.append(player_data)

        

    # Concatenate all players' enriched data for this position
    full_position_data = pd.concat(all_players_data, ignore_index=True)
    

    # Create csv for each GW+position
    for gw in full_position_data['GW'].unique():
        gw_data = full_position_data[full_position_data['GW'] == gw].copy()
        gw_data = gw_data[subset_cols]

        # set output path
        output_filename = f"{position}_GW_{gw}_players.csv"
        output_path = f'/Users/alexroberts/Documents/Diss (new)/(1) Making csv/Individual GW+position files/{output_filename}'
        gw_data.to_csv(output_path, index=False)
        

print('Done')  

Done
